In [1]:
from dotenv import load_dotenv
from groq import Groq
import os
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [2]:
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)

In [3]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages = messages )
response.choices[0].message.content

'Absolutely! I’d be happy to help you get started. To make sure I give you the most accurate information, could you let me know which specific course you’re interested in? \n\nIf you already have a course name or code, just share that, and I can walk you through:\n\n1. **Eligibility & Prerequisites** – Any required background or prior courses.\n2. **Enrollment Steps** – How to register, any deadlines, and where to sign up.\n3. **Course Details** – Duration, format (self‑paced, live sessions, hybrid), and any required materials.\n4. **Support & Resources** – Where to find help if you run into any issues.\n\nOnce I have a bit more detail, I’ll give you a step‑by‑step guide to join the course right away. Looking forward to hearing from you!'

Define search tool

In [4]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [5]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

The description is the most important field, because the model reads it to decide when to call the function. parameters is a JSON schema for the arguments, and we mark query as required so the model always fills it in.

Sending the question with the tool

In [6]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=[search_tool],
)

response

ChatCompletion(id='chatcmpl-c4a22245-8a54-4003-b750-0ffb52529c02', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user asks: "I just discovered the course. Can I join it?" We need to answer according to the course FAQ. Likely answer: Yes, you can join as a participant; registration details: open to anyone, just sign up, etc. Provide information about enrollment, prerequisites, cost, etc. The policy: provide info based on FAQ. Might need to search FAQ. Let\'s search.', tool_calls=[ChatCompletionMessageToolCall(id='fc_94a6042e-f830-4435-b126-957b94e3a3cd', function=Function(arguments='{"query":"join course enrollment"}', name='search'), type='function')]))], created=1782044413, model='openai/gpt-oss-120b', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_803c0ba83d', usage=CompletionUsag

To check if tool was called

In [7]:
# Check if tool was called
if response.choices[0].message.tool_calls:
    print("Tool was called!")
    # Access the tool call details
    tool_call = response.choices[0].message.tool_calls[0]
    print(f"Tool name: {tool_call.function.name}")
    print(f"Arguments: {tool_call.function.arguments}")
else:
    print("No tool was used.")

Tool was called!
Tool name: search
Arguments: {"query":"join course enrollment"}


The LLM is a text model - it can suggest calling a function but can't actually run Python code

It can only describe what function to call and with what arguments

Executing the function and sending the result back

In [8]:
import json

# Extract the tool call
tool_call = response.choices[0].message.tool_calls[0]
args = json.loads(tool_call.function.arguments)

# Execute the search function
results = search(**args)
result_json = json.dumps(results, indent=2)

In [9]:
# Append the assistant's tool call to messages
messages.append({
    "role": "assistant",
    "content": None,
    "tool_calls": response.choices[0].message.tool_calls
})

# Append the tool result
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result_json
})

In [10]:
# Get final response
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=[search_tool],
)
response.choices[0].message.content

'Yes—you can still join the LLM\u202fZoomcamp. The course is open‑ended, so you can start learning right away and submit homework while the enrollment form is still open.  \n\nIf you want a certificate, you’ll need to finish the “live” cohort (i.e., submit your capstone project before the submission deadline and complete the required peer‑reviews). Otherwise, you can simply work through the materials at your own pace.'

Flow:

User: "I just discovered the course. Can I join it?"

Model: "I should search the FAQ" → Returns tool call

Your code: Executes search("join course enrollment") → Gets FAQ entries

Your code: Sends results back to model

Model: Reads FAQ entries and gives accurate answer about enrollment process

# Agentic Loop

We sent a message and got back a function call. We ran it, sent the result back, and got the answer.

That works for one function call. It breaks down when the model wants to search several times, or when the first search misses the answer. We don't know in advance how many calls the model will want. So we need a loop that keeps calling the model and running tools until it's done. An agent is exactly that.

So far we've relied on the model to figure out when to search. We make that more reliable with a developer message that spells out how to behave. This is where we give the agent its role. The same message also pushes it toward multiple searches, so we get to watch the loop run more than once.

In [11]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

A function-call helper

We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.

In [20]:
def make_call(call):
    args = json.loads(call.function.arguments)
    
    if call.function.name == "search":
        result = search(**args)
    else:
        result = {"error": f"Unknown function: {call.function.name}"}
    
    result_json = json.dumps(result, indent=2)
    
    # Return with proper OpenAI/Groq format
    return {
        "role": "tool",          
        "tool_call_id": call.id,
        "content": result_json,
    }

In [15]:
response.choices[0].message.tool_calls

[ChatCompletionMessageToolCall(id='fc_1610f8dc-1dc1-4a98-8984-cb81cfd2440a', function=Function(arguments='{"query":"can I join the course enrollment eligibility"}', name='search'), type='function')]

In [21]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
    tools=[search_tool],
)

has_function_calls = False

# Check if there are tool calls
if response.choices[0].message.tool_calls:
    # There are tool calls - process them
    has_function_calls = True
    
    # First, add the assistant's tool call to the messages
    messages.append({
        "role": "assistant",
        "content": None,
        "tool_calls": response.choices[0].message.tool_calls
    })
    
    # Then process each tool call
    for tool_call in response.choices[0].message.tool_calls:
        print(f"function_call: {tool_call.function.name} - args: {tool_call.function.arguments}")
        call_output = make_call(tool_call)
        messages.append(call_output)  # Append the result
    
    # Get final response with tool results
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        tools=[search_tool],
    )
    
    print("\nFINAL ANSWER:")
    print(response.choices[0].message.content)
    
else:
    # No tool calls - just print the direct response
    print("No tool calls needed. Direct response:")
    print(response.choices[0].message.content)

function_call: search - args: {"query":"Can I join the course? enrollment eligibility FAQ"}

FINAL ANSWER:
**Can you still join the LLM Zoomcamp?**  
Yes! The course is open for new participants even after it has started. Here’s what you need to know:

| What you want to do | What’s required | How to get started |
|----------------------|----------------|--------------------|
| **Just learn the material** | None | You can immediately start watching the videos, cloning the GitHub repo, and working through the notebooks. No registration or approval is needed. |
| **Earn a certificate** | • Submit a final project **while the submission window is open**  <br>• Complete the required **peer‑reviews (3 capstone reviews)**  <br>• Be part of a **live cohort** (the self‑paced track does **not** award certificates) | 1. Register on the course platform (this is only to gauge interest; you’ll still be accepted).  <br>2. Follow the weekly workflow (watch lessons, do the homework, push your code).  <

Wrapping it in a function

In [22]:
def agent_loop(instructions, question, model="openai/gpt-oss-120b", max_iterations=5, verbose=True) -> str:
    """
    Run the agent loop with tool calling capabilities.
    """
    
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]
    
    iteration = 1
    last_answer = None
    
    while iteration <= max_iterations:
        if verbose:
            print(f"\n{'='*50}")
            print(f"🔄 Iteration #{iteration}")
            print(f"{'='*50}")
        
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )
        
        # Check for tool calls
        if response.choices[0].message.tool_calls:
            if verbose:
                print(f"🔧 Tool calls detected:")
            
            # Add assistant's tool call to messages
            messages.append({
                "role": "assistant",
                "content": None,
                "tool_calls": response.choices[0].message.tool_calls
            })
            
            # Process each tool call
            for tool_call in response.choices[0].message.tool_calls:
                if verbose:
                    print(f"  Function: {tool_call.function.name}")
                    print(f"  Arguments: {tool_call.function.arguments}")
                
                # Execute the function
                args = json.loads(tool_call.function.arguments)
                if tool_call.function.name == "search":
                    result = search(**args)
                else:
                    result = {"error": f"Unknown function: {tool_call.function.name}"}
                
                # Add result to messages
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(result, indent=2)
                })
            
            iteration += 1
            
        else:
            # No tool calls - final answer
            last_answer = response.choices[0].message.content
            if verbose:
                print(f"Final answer ready")
                print(f"{'='*50}")
                print(f"ASSISTANT: {last_answer}")
            break
    
    if last_answer is None:
        last_answer = f"Maximum iterations ({max_iterations}) reached without getting a final answer."
    
    return last_answer

In [23]:
agent_loop(instructions, "How do I run Olama locally?")


🔄 Iteration #1
🔧 Tool calls detected:
  Function: search
  Arguments: {"query":"Olama locally"}

🔄 Iteration #2
🔧 Tool calls detected:
  Function: search
  Arguments: {"query":"Olama"}

🔄 Iteration #3
🔧 Tool calls detected:
  Function: search
  Arguments: {"query":"Ollama run locally"}

🔄 Iteration #4
Final answer ready
ASSISTANT: Below is a quick‑start guide to get **Ollama** up and running on your own machine.  
It covers the three things you need to do:

1. **Install the Ollama server** (the binary that talks to the model).  
2. **Pull + run a model** (e.g., Llama 3).  
3. **Talk to it from the command line or from Python**.

---

## 1️⃣ Install Ollama

| OS | Command / steps |
|----|-----------------|
| **macOS** | Download the `.pkg` from <https://ollama.com/download> and double‑click to install. |
| **Windows** | Download the `.msi` from the same page and run the installer. |
| **Linux** | Run the one‑liner in a terminal: <br>`curl -fsSL https://ollama.com/install.sh \| sh` |
| 

'Below is a quick‑start guide to get **Ollama** up and running on your own machine.  \nIt covers the three things you need to do:\n\n1. **Install the Ollama server** (the binary that talks to the model).  \n2. **Pull\u202f+\u202frun a model** (e.g., Llama\u202f3).  \n3. **Talk to it from the command line or from Python**.\n\n---\n\n## 1️⃣ Install Ollama\n\n| OS | Command / steps |\n|----|-----------------|\n| **macOS** | Download the\u202f`.pkg` from <https://ollama.com/download> and double‑click to install. |\n| **Windows** | Download the\u202f`.msi` from the same page and run the installer. |\n| **Linux** | Run the one‑liner in a terminal: <br>`curl -fsSL https://ollama.com/install.sh \\| sh` |\n| **WSL / Docker** | You can also run the official Docker image (`ollama/ollama`) – see the “Docker” note at the bottom. |\n\n> After the installer finishes you should have an `ollama` executable on your `$PATH`. Test it:\n\n```bash\nollama --version\n# e.g. ollama version 0.1.35\n```\n\n---\

In [24]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")


🔄 Iteration #1
🔧 Tool calls detected:
  Function: search
  Arguments: {"query":"Can I still join the course after it started"}

🔄 Iteration #2
Final answer ready
ASSISTANT: **Yes – you can still join the LLM Zoomcamp even if you discovered it after the start date.**  

- **Start right away** – The course materials (videos, notebooks, and the GitHub repo) are publicly available, so you can begin learning immediately.  
- **Registration** – It’s mainly for gauging interest; you don’t need a confirmation email or a formal enrollment step to start working on the lessons or homework.  
- **Getting a certificate** – The only requirement for a certificate is that you **submit a completed capstone project while the submission window is still open** (and later complete the required three peer‑reviews). Completing the weekly homework is helpful but not mandatory for the certificate.

So feel free to dive in, complete the assignments that interest you, and make sure you submit your final project

'**Yes – you can still join the LLM\u202fZoomcamp even if you discovered it after the start date.**  \n\n- **Start right away** – The course materials (videos, notebooks, and the GitHub repo) are publicly available, so you can begin learning immediately.  \n- **Registration** – It’s mainly for gauging interest; you don’t need a confirmation email or a formal enrollment step to start working on the lessons or homework.  \n- **Getting a certificate** – The only requirement for a certificate is that you **submit a completed capstone project while the submission window is still open** (and later complete the required three peer‑reviews). Completing the weekly homework is helpful but not mandatory for the certificate.\n\nSo feel free to dive in, complete the assignments that interest you, and make sure you submit your final project before the deadline if you want the certification.\n\n---\n\nIs there anything else you’d like to know—e.g., how the weekly workflow works, where to find the pro

Right now the agent will answer anything. Ask it about chess and it will still try.

In [25]:
agent_loop(instructions, "what's queen gambit?")


🔄 Iteration #1
🔧 Tool calls detected:
  Function: search
  Arguments: {"query":"queen gambit definition"}

🔄 Iteration #2
Final answer ready
ASSISTANT: **The Queen’s Gambit (often written “Queen’s Gambit”) is a classic chess opening.**  

---

## 1. Basic definition  

- **Opening moves:**  
  1. d4 d5 2. c4  

- White offers the pawn on **c4** to Black. The idea is to tempt Black into accepting the pawn (the “gambit”) and then use the resulting central control and piece activity to gain an advantage.  

- The opening belongs to the **“closed”** family of openings (because it starts with 1.d4) and is one of the oldest and most studied openings in chess history, dating back to the 15th‑century.

---

## 2. Main ideas behind the gambit  

| For White | For Black |
|-----------|-----------|
| • **Control the center** with the d‑ and e‑pawns after the pawn exchange.<br>• Open the **c‑file** for the queen or a rook.<br>• Develop pieces quickly (Nf3, e3, Nc3, Bd3, etc.). | • **Hold onto the

'**The Queen’s Gambit (often written “Queen’s Gambit”) is a classic chess opening.**  \n\n---\n\n## 1. Basic definition  \n\n- **Opening moves:**  \n  1.\u202fd4\u202fd5\u202f2.\u202fc4  \n\n- White offers the pawn on **c4** to Black. The idea is to tempt Black into accepting the pawn (the “gambit”) and then use the resulting central control and piece activity to gain an advantage.  \n\n- The opening belongs to the **“closed”** family of openings (because it starts with 1.d4) and is one of the oldest and most studied openings in chess history, dating back to the 15th‑century.\n\n---\n\n## 2. Main ideas behind the gambit  \n\n| For White | For Black |\n|-----------|-----------|\n| • **Control the center** with the d‑ and e‑pawns after the pawn exchange.<br>• Open the **c‑file** for the queen or a rook.<br>• Develop pieces quickly (Nf3, e3, Nc3, Bd3, etc.). | • **Hold onto the extra pawn** (if they accept – 2…dxc4).<br>• Return the pawn later (often with …b5‑…b4 or …e6‑…a6‑…b5) to comple

We want a course assistant, not a general chatbot. We tighten the instructions so the agent only answers from the FAQ. For our own use we might be fine letting it answer from general knowledge

In [26]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")


🔄 Iteration #1
🔧 Tool calls detected:
  Function: search
  Arguments: {"query":"queen gambit"}

🔄 Iteration #2
Final answer ready
ASSISTANT: I’m sorry, but I don’t have information on that topic in the course FAQ. Is there something else about the course or its logistics I can help you with?


'I’m sorry, but I don’t have information on that topic in the course FAQ. Is there something else about the course or its logistics I can help you with?'

# ToyAIkit
The handwritten agent loop is repetitive. Every time you build a new agent, you'd write the same while-loop, the same function-call handling, the same message management.

ToyAIKit wraps this pattern so you can focus on tools, prompts, and behavior. It is experimentation library built by DataTalksCub , and it is NOT meant for production use. 

# Other Frameworks
## OpenAI Agents SDK
The official SDK from OpenAI for building agents. It uses the same Responses API we used throughout this module. It supports tool definition, multi-turn conversations, and handoffs between agents.
`uv add openai-agents`

## PydanticAI
A type-safe agent framework that supports multiple LLM providers. Tools are plain Python functions with type hints, no wrappers needed. Switching providers is as simple as changing the model string.
`uv add pydantic-ai`

## LangChain / LangGraph
A popular framework with lots of integrations. LangChain handles the basics, and LangGraph adds graph-based workflows for more complex agent patterns.

Good choice if you need lots of integrations (vector stores, document loaders, etc.) and a large community.

## Google ADK
The Agent Development Kit from Google. If you plan to use Gemini models, this is the one I'd reach for. It exposes the same building blocks we've seen, like tools, instructions, and sessions. It also integrates with Google Cloud.

Good choice if your stack is on Google Cloud or you specifically want Gemini.

## Others

Other frameworks worth knowing:

- CrewAI - multi-agent orchestration
- AutoGen - multi-agent conversations from Microsoft
- Semantic Kernel - from Microsoft, supports C# and Python
- Smolagents - lightweight agent framework from HuggingFace
- Anthropic Tool Use - Anthropic's native tool use API

In [27]:
!uv add toyaikit

Resolved 128 packages in 6.05s
Prepared 7 packages in 11.61s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 7 packages in 2.19s
 + anthropic==0.111.0
 + docstring-parser==0.18.0
 + genai-prices==0.0.66
 + httpcore2==2.4.0
 + httpx2==2.4.0
 + toyaikit==0.0.11
 + truststore==0.10.4


In [34]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

## Registering the tool

In [29]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

## Letting ToyAIKit generate the schema

In [30]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

Then register it without passing a schema:

In [31]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [32]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

## The chat interface and runner
Create the chat interface and a callback, then build the runner:

In [38]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [39]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

You can use OpenAIChatCompletionsClient with a custom OpenAI client configured for Groq's base URL.

In [41]:
from openai import OpenAI
from toyaikit.llm import OpenAIChatCompletionsClient
from toyaikit.chat.runners import OpenAIChatCompletionsRunner
from toyaikit.chat import IPythonChatInterface
from toyaikit.tools import Tools
import os

In [42]:
# Initialize the OpenAI client for Groq
groq_openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"  # Groq's OpenAI-compatible endpoint
)

In [ ]:
# Create the ToyAIKit LLM client
llm_client = OpenAIChatCompletionsClient(
    model="openai/gpt-oss-120b",  
    client=groq_openai_client
)

In [44]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [45]:
chat_interface = IPythonChatInterface()


runner = OpenAIChatCompletionsRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client  
)

In [46]:
#Run the Agent
runner.run()

-> Response received


-> Response received


e:\Repos\DTC\llm\01-agentic-rag\.venv\Lib\site-packages\toyaikit\chat\runners.py:542: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost = self.pricing_config.calculate_cost(


Chat ended.


e:\Repos\DTC\llm\01-agentic-rag\.venv\Lib\site-packages\toyaikit\chat\runners.py:184: UnknownModelWarning: No pricing data for model 'openai/gpt-oss-120b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  combined_cost = self.pricing_config.calculate_cost(


LoopResult(new_messages=[{'role': 'system', 'content': "You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore."}, {'role': 'user', 'content': 'Can I join the course?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionT